# Engine Testing

Tries out routing, priority scoring, duplicate detection and the full complaint engine.

- **Sections 1 to 3** need no trained models and run quickly.
- **Sections 4 to 6** load MuRIL and EfficientNet, so train `MuRIL_urgency.ipynb` first and close other notebooks to save memory.

Run from the `notebooks/` folder.

## Section 0: Setup

### 0.1 Make the project importable
Adds `src/` to the import path so the notebook can import the project modules without installing the package. Autoreload picks up edits to the `.py` files without restarting the kernel. Only the lightweight modules are imported here; the engine (which loads PyTorch) comes later. Use `"."` instead of `".."` if the notebook is in the project root.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, glob, random
from datetime import datetime, timedelta
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

from civicai_finalyearproject import config as cfg, routing, priority, duplicates

pd.set_option("display.max_colwidth", 80)
print("Project root:        ", cfg.PROJECT_ROOT)
print("Category model found:", os.path.isdir(cfg.CATEGORY_MODEL_DIR))
print("Urgency model found: ", os.path.isdir(cfg.URGENCY_MODEL_DIR))
print("Image model found:   ", os.path.isfile(cfg.IMAGE_MODEL_PATH))

## Section 1: Test Routing (no models needed)

### 1.1 Check that every label maps to a department
Lists every text category and image class with the department it routes to. Anything showing "Unassigned" needs a keyword rule added in `routing.py`.

In [ ]:
text_labels = sorted(pd.read_csv(os.path.join(cfg.TEXT_DATA_DIR, "train.csv"))["category"].unique())
image_labels = sorted(os.listdir(os.path.join(cfg.IMAGE_DATA_DIR, "train")))

print("TEXT LABELS")
display(pd.DataFrame(routing.mapping_table(text_labels), columns=["text label", "department"]))
print("IMAGE LABELS")
display(pd.DataFrame(routing.mapping_table(image_labels), columns=["image label", "department"]))

### 1.2 Test the decision rule with made-up predictions
Feeds the routing rule fake predictions for each situation: agreement, disagreement, weak image, text only, image only and nothing. Confirm that each row's department, source and review flag match what you expect.

In [ ]:
cases = {
    "image confident, text agrees":        ({"label": "trash", "confidence": 0.95},    {"label": "Garbage_Waste_Trash", "confidence": 0.98}),
    "image confident, text disagrees":     ({"label": "drainage", "confidence": 0.92}, {"label": "Road_Issues_Pothole", "confidence": 0.97}),
    "image weak -> text fallback":         ({"label": "pothole", "confidence": 0.90},  {"label": "Vandalism_Graffiti", "confidence": 0.40}),
    "text only":                           ({"label": "drainage", "confidence": 0.88}, None),
    "image only (confident)":              (None, {"label": "Road_Issues_Pothole", "confidence": 0.96}),
    "text only, low confidence":           ({"label": "trash", "confidence": 0.35},    None),
    "nothing":                             (None, None),
}

rows = []
for name, (t, i) in cases.items():
    d = routing.route(t, i)
    rows.append({"case": name, "department": d.department, "source": d.source,
                 "needs_review": d.needs_review, "reason": d.reason})
pd.DataFrame(rows)

## Section 2: Test the Priority Formula (no models needed)

### 2.1 See how duplicates change priority
Shows the final score and priority level for each urgency level with 0 to 5 duplicate reports. Check that the cap works (many duplicates cannot push a low-urgency issue above a single high one) and that the levels look sensible. Adjust the values in `config.py` if not.

If `urgency_stats.json` does not exist yet, default base scores (2 / 5 / 8) are used.

In [ ]:
stats = priority.load_stats()
print("Base scores in use:", stats["level_base_score"])

rows = []
for level in ["low", "medium", "high"]:
    base = stats["level_base_score"][level]
    for dups in [0, 1, 2, 3, 5]:
        score = priority.compute_priority(base, dups)
        rows.append({"urgency": level, "base score": base, "duplicates": dups,
                     "priority score": score, "priority level": priority.priority_level(score)})
pd.DataFrame(rows)

## Section 3: Test Duplicate Detection (no models needed)

### 3.1 Compare a new complaint with existing ones
The new complaint is a romanized rewording of existing complaint 1, near the same spot. Complaint 2 is a different department, and complaint 3 is the same department but about 2 km away. Only complaint 1 should match.

The printed text similarities are what you tune `TEXT_SIM_THRESHOLD` against.

In [ ]:
now = datetime.now()
existing = [
    {"id": 1, "description": "road bigreko chha mhepi najikai", "department": routing.DEPT_ROADS,
     "lat": 27.7172, "lon": 85.3240, "timestamp": now - timedelta(hours=5), "status": "open"},
    {"id": 2, "description": "garbage overflowing near the market", "department": routing.DEPT_WASTE,
     "lat": 27.7173, "lon": 85.3241, "timestamp": now - timedelta(hours=3), "status": "open"},
    {"id": 3, "description": "road damaged near mhepi, dangerous", "department": routing.DEPT_ROADS,
     "lat": 27.7350, "lon": 85.3400, "timestamp": now - timedelta(hours=2), "status": "open"},
]
new = {"description": "mhepi najik road bigriyeko chha", "department": routing.DEPT_ROADS,
       "lat": 27.7174, "lon": 85.3242, "timestamp": now, "image": None}

sims = duplicates.text_similarities(new["description"], [c["description"] for c in existing])
print("Text similarity to each existing complaint:", [round(s, 3) for s in sims])
print("Threshold:", cfg.TEXT_SIM_THRESHOLD)
print("\nMatches:", duplicates.find_duplicates(new, existing))

## Section 4: Test the Full Engine (loads the models)

### 4.1 Import the engine and run a romanized Nepali complaint
This is the first cell that loads PyTorch and MuRIL, so it takes a while on the first run. It uses the example from the start of the project. The result shows category, department, urgency, duplicates and priority.

In [ ]:
from civicai_finalyearproject import engine

SHOW = ["category", "department", "routing_source", "needs_review", "routing_reason",
        "urgency_level", "urgency_source", "base_score", "duplicate_count",
        "priority_score", "priority_level"]

r = engine.process_complaint(text="road bigreko chha mhepi najikai", lat=27.7172, lon=85.3240)
pd.Series({k: r[k] for k in SHOW})

### 4.2 Helper to pick a random test image
Picks a random photo from the image test split (optionally from one class) so you can try image submissions without searching for files.

In [ ]:
def sample_image(class_folder=None):
    root = os.path.join(cfg.IMAGE_DATA_DIR, "test")
    folder = class_folder or random.choice(os.listdir(root))
    path = random.choice(glob.glob(os.path.join(root, folder, "*")))
    return path, folder

img_path, folder = sample_image("Road_Issues_Pothole")
print("Using:", folder, img_path)

### 4.3 Image only, and text plus image
An image-only complaint has no text, so urgency should come from `category_default`. Then the same photo with a romanized description should use the urgency model instead.

In [ ]:
r_img = engine.process_complaint(image=img_path, lat=27.7172, lon=85.3240)
print("IMAGE ONLY")
display(pd.Series({k: r_img[k] for k in SHOW}))

r_both = engine.process_complaint(text="road bigreko chha mhepi najikai", image=img_path, lat=27.7172, lon=85.3240)
print("TEXT + IMAGE")
display(pd.Series({k: r_both[k] for k in SHOW}))

### 4.4 Disagreement case
A pothole-style description with a garbage photo. The image is trusted, but the complaint should be flagged for review because the two departments differ.

In [ ]:
garbage_path, _ = sample_image("Garbage_Waste_Trash")
r_conflict = engine.process_complaint(text="road bigreko chha, pothole ma gadi fasyo", image=garbage_path,
                                      lat=27.7172, lon=85.3240)
pd.Series({k: r_conflict[k] for k in SHOW})

## Section 5: Simulate a Stream of Complaints

### 5.1 Several reports of the same pothole
Processes four complaints in time order, adding each to a list that acts as the database. The first three describe the same road problem in different words and should accumulate duplicates and a rising priority. The fourth is a garbage complaint nearby and should stay separate.

If the reports are not recognised as duplicates, check the similarity values from section 3 and lower `TEXT_SIM_THRESHOLD` in `config.py`.

In [ ]:
now = datetime.now()
reports = [
    ("road bigreko chha mhepi najikai",                          27.7172, 85.3240, 6),
    ("mhepi najik road ma dhulo ra dhaka bigriyeko chha",        27.7173, 85.3241, 4),
    ("road damaged near mhepi, very dangerous for bikes",        27.7174, 85.3242, 2),
    ("ghar ko fohor sadak ma overflow bhayeko chha",             27.7175, 85.3243, 1),
]

database = []
for i, (text, lat, lon, hours_ago) in enumerate(reports, start=1):
    result = engine.process_complaint(
        text=text, lat=lat, lon=lon,
        timestamp=now - timedelta(hours=hours_ago),
        existing_complaints=database, complaint_id=i,
    )
    database.append(result)

pd.DataFrame(database)[["id", "description", "department", "urgency_level", "base_score",
                        "duplicate_count", "priority_score", "priority_level", "needs_review"]]

## Section 6: Measure Routing Accuracy

### 6.1 Compare text-only, image-only and combined routing
Each test case has a description, an optional photo and the correct department. The harness routes every case three ways and reports accuracy for each, which is the evidence for the image-first design.

The starter list is small. Extend it to 30 to 50 cases, including drainage (which has no image class), romanized and Devanagari text, and some cases with no photo.

In [ ]:
def eval_routing(cases):
    results = {"text_only": [], "image_only": [], "combined": []}
    for c in cases:
        expected, text, img = c["expected_department"], c.get("text"), c.get("image_path")
        if text:
            results["text_only"].append(engine.analyze(text=text)["routing"].department == expected)
        if img:
            results["image_only"].append(engine.analyze(image=img)["routing"].department == expected)
        if text and img:
            results["combined"].append(engine.analyze(text=text, image=img)["routing"].department == expected)
    return pd.DataFrame([
        {"setup": k, "cases": len(v), "correct": sum(v),
         "accuracy": round(sum(v) / len(v), 3) if v else None}
        for k, v in results.items()
    ])

eval_cases = [
    {"text": "mhepi najikai road bigreko chha", "image_path": sample_image("Road_Issues_Pothole")[0],
     "expected_department": routing.DEPT_ROADS},
    {"text": "ghar ko fohor sadak ma overflow bhayeko chha", "image_path": sample_image("Garbage_Waste_Trash")[0],
     "expected_department": routing.DEPT_WASTE},
    {"text": "gadi galat thau ma park gareko chha, bato block bhayo",
     "image_path": sample_image("Parking_Issues_Illegal_Parking")[0],
     "expected_department": routing.DEPT_TRAFFIC},
    {"text": "bhitta ma graffiti lekhiyeko chha", "image_path": sample_image("Vandalism_Graffiti")[0],
     "expected_department": routing.DEPT_WARD},
    {"text": "traffic sign tutera jhukeko chha", "image_path": sample_image("Road_Issues_Damaged_Sign")[0],
     "expected_department": routing.DEPT_ROADS},
    {"text": "nali jam bhayera pani sadak ma aayeko chha", "image_path": None,
     "expected_department": routing.DEPT_DRAINAGE},
]
eval_routing(eval_cases)